# PLS (Partial Least Squares) Regression - CUCOMOF Dataset
Dự đoán nồng độ glucose (mM) từ cường độ dòng điện (µA) tại các mốc điện thế.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict, LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

print('Libraries loaded!')

## 1. Load Data

In [ ]:
df = pd.read_csv('CUCOMOF_processed.csv')
print(f'Shape: {df.shape}')
print(f'Số mẫu: {len(df)}')
print(f'Số features: {len(df.columns) - 1}')
df

In [ ]:
# Tách X và y
X = df.drop(columns=['glucose_mM']).values
y = df['glucose_mM'].values
feature_names = df.drop(columns=['glucose_mM']).columns.tolist()

print(f'X shape: {X.shape}')
print(f'y: {y}')

## 2. Chọn số Components tối ưu

In [ ]:
# Scale X
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dùng Leave-One-Out CV để chọn số components tối ưu
loo = LeaveOneOut()
max_components = min(len(df) - 1, X.shape[1])  # tối đa n-1 hoặc số features

rmse_cv = []
r2_cv = []

for n_comp in range(1, max_components + 1):
    pls = PLSRegression(n_components=n_comp)
    y_pred_cv = cross_val_predict(pls, X_scaled, y, cv=loo)
    rmse = np.sqrt(mean_squared_error(y, y_pred_cv))
    r2 = r2_score(y, y_pred_cv)
    rmse_cv.append(rmse)
    r2_cv.append(r2)
    print(f'n_components={n_comp}: RMSE={rmse:.4f}, R²={r2:.4f}')

best_n = np.argmin(rmse_cv) + 1
print(f'\n✅ Số components tối ưu: {best_n} (RMSE={rmse_cv[best_n-1]:.4f})')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

components = range(1, max_components + 1)

ax1.plot(components, rmse_cv, 'bo-', linewidth=2, markersize=8)
ax1.axvline(x=best_n, color='red', linestyle='--', label=f'Best = {best_n}')
ax1.set_xlabel('Số Components')
ax1.set_ylabel('RMSE (LOO-CV)')
ax1.set_title('RMSE vs Số Components')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(components, r2_cv, 'go-', linewidth=2, markersize=8)
ax2.axvline(x=best_n, color='red', linestyle='--', label=f'Best = {best_n}')
ax2.set_xlabel('Số Components')
ax2.set_ylabel('R² (LOO-CV)')
ax2.set_title('R² vs Số Components')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pls_components_selection.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Huấn luyện mô hình PLS tối ưu

In [ ]:
# Train model với số components tối ưu
pls_best = PLSRegression(n_components=best_n)
pls_best.fit(X_scaled, y)

# Dự đoán train set
y_pred_train = pls_best.predict(X_scaled).ravel()

# Dự đoán LOO-CV
y_pred_cv = cross_val_predict(pls_best, X_scaled, y, cv=LeaveOneOut()).ravel()

# Metrics
r2_train = r2_score(y, y_pred_train)
rmse_train = np.sqrt(mean_squared_error(y, y_pred_train))
r2_loo = r2_score(y, y_pred_cv)
rmse_loo = np.sqrt(mean_squared_error(y, y_pred_cv))

print('=== Kết quả mô hình PLS ===')
print(f'Số components: {best_n}')
print(f'\nTrain Set:')
print(f'  R²     = {r2_train:.4f}')
print(f'  RMSE   = {rmse_train:.4f} mM')
print(f'\nLOO Cross-Validation:')
print(f'  R²     = {r2_loo:.4f}')
print(f'  RMSE   = {rmse_loo:.4f} mM')

## 4. Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Plot 1: Predicted vs Actual ---
ax = axes[0]
ax.scatter(y, y_pred_cv, color='steelblue', s=100, zorder=5, label='LOO-CV Predicted')
ax.scatter(y, y_pred_train, color='orange', s=60, marker='x', zorder=6, label='Train Predicted')

# Đường lý tưởng y=x
lim = [min(y.min(), y_pred_cv.min()) - 0.2, max(y.max(), y_pred_cv.max()) + 0.2]
ax.plot(lim, lim, 'r--', linewidth=1.5, label='Ideal')

for i, (actual, pred) in enumerate(zip(y, y_pred_cv)):
    ax.annotate(f'{actual}mM', (actual, pred), textcoords='offset points', xytext=(5, 5), fontsize=8)

ax.set_xlabel('Actual Glucose (mM)')
ax.set_ylabel('Predicted Glucose (mM)')
ax.set_title(f'Predicted vs Actual\nR²(LOO)={r2_loo:.4f}, RMSE={rmse_loo:.4f} mM')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 2: Residuals ---
ax2 = axes[1]
residuals = y_pred_cv - y
ax2.bar(y, residuals, width=0.1, color=['red' if r < 0 else 'steelblue' for r in residuals], alpha=0.7)
ax2.axhline(y=0, color='black', linewidth=1)
ax2.set_xlabel('Actual Glucose (mM)')
ax2.set_ylabel('Residual (Predicted - Actual)')
ax2.set_title('Residuals (LOO-CV)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pls_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. VIP Scores (Variable Importance in Projection)

In [ ]:
def vip_scores(pls_model):
    """Tính VIP scores cho PLS model"""
    t = pls_model.x_scores_
    w = pls_model.x_weights_
    q = pls_model.y_loadings_
    p, h = w.shape
    vips = np.zeros((p,))
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)
    for i in range(p):
        weight = np.array([(w[i, j] / np.linalg.norm(w[:, j]))**2 for j in range(h)])
        vips[i] = np.sqrt(p * (s.T @ weight) / total_s)
    return vips

vip = vip_scores(pls_best)

plt.figure(figsize=(10, 4))
colors = ['red' if v >= 1.0 else 'steelblue' for v in vip]
bars = plt.bar(feature_names, vip, color=colors, alpha=0.8, edgecolor='black')
plt.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, label='VIP = 1.0 (threshold)')
plt.xlabel('Features (Voltage)')
plt.ylabel('VIP Score')
plt.title('VIP Scores - Feature Importance')
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('pls_vip_scores.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nVIP Scores:')
for name, score in zip(feature_names, vip):
    marker = '⭐' if score >= 1.0 else '  '
    print(f'  {marker} {name}: {score:.4f}')

## 6. PLS Components Scores Plot

In [ ]:
if best_n >= 2:
    T = pls_best.x_scores_
    plt.figure(figsize=(7, 5))
    scatter = plt.scatter(T[:, 0], T[:, 1], c=y, cmap='viridis', s=150, edgecolor='black', zorder=5)
    for i, conc in enumerate(y):
        plt.annotate(f'{conc}mM', (T[i, 0], T[i, 1]), textcoords='offset points', xytext=(8, 4), fontsize=9)
    plt.colorbar(scatter, label='Glucose (mM)')
    plt.xlabel('PLS Component 1')
    plt.ylabel('PLS Component 2')
    plt.title('PLS Scores Plot (T1 vs T2)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('pls_scores_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Chỉ có 1 component, không vẽ được scores plot 2D.')

## 7. Tổng kết kết quả

In [ ]:
results = pd.DataFrame({
    'Actual (mM)': y,
    'Predicted LOO-CV (mM)': y_pred_cv.round(4),
    'Residual': (y_pred_cv - y).round(4),
    'Abs Error': np.abs(y_pred_cv - y).round(4)
})

print('=== Bảng kết quả dự đoán ===')
print(results.to_string(index=False))
print(f'\nMean Abs Error: {results["Abs Error"].mean():.4f} mM')
print(f'R² (LOO-CV):   {r2_loo:.4f}')
print(f'RMSE (LOO-CV): {rmse_loo:.4f} mM')